# PY200 — GST (Goods and Services Tax) Calculation

A practical exercise that demonstrates **`match...case`** (Python 3.10+), **`if/elif/else`**, and real-world tax bracket logic.

## India's GST Slabs (Simplified)

| Slab | Rate | Example Items |
|---|---|---|
| Essential | 0% | Fresh fruits, vegetables, milk, bread |
| Low | 5% | Packaged food, economy hotels, transport |
| Standard | 12% | Processed food, business-class hotels, apparel (< ₹1000) |
| Higher | 18% | Electronics, restaurants, IT services, financial services |
| Luxury | 28% | Luxury cars, tobacco, aerated drinks, cement |

> **Perspective:** Tax calculation is a great example of **business rule logic** — the kind of code that drives ERP systems, e-commerce platforms, and billing software worldwide.

---
## Approach 1: Using `match...case` (Python 3.10+)

The `match` statement provides clean, readable branching — ideal when matching against known categories.

In [ ]:
## GST Calculation using match...case

def calculate_gst_match(item_category: str, base_price: float) -> dict:
    """
    Calculate GST based on item category using match...case.
    Returns a dictionary with the breakdown.
    """
    # Normalize input to lowercase for consistent matching
    match item_category.lower():
        case "essential":
            rate = 0.0
        case "low":
            rate = 0.05
        case "standard":
            rate = 0.12
        case "higher":
            rate = 0.18
        case "luxury":
            rate = 0.28
        case _:
            # Wildcard _ handles any unrecognized category
            print(f"Warning: Unknown category '{item_category}', defaulting to 18%")
            rate = 0.18

    gst_amount = base_price * rate
    total_price = base_price + gst_amount

    return {
        "category": item_category,
        "base_price": base_price,
        "gst_rate": f"{rate * 100:.0f}%",
        "gst_amount": round(gst_amount, 2),
        "total_price": round(total_price, 2)
    }


# Test with different categories
items = [
    ("Essential", 200.0),
    ("Low", 500.0),
    ("Standard", 1200.0),
    ("Higher", 25000.0),
    ("Luxury", 150000.0),
]

for category, price in items:
    result = calculate_gst_match(category, price)
    print(f"{result['category']:>10} | Base: ₹{result['base_price']:>10,.2f} | "
          f"GST ({result['gst_rate']:>3}): ₹{result['gst_amount']:>10,.2f} | "
          f"Total: ₹{result['total_price']:>10,.2f}")

---
## Approach 2: Using `if/elif/else`

Works on **all Python versions**. Functionally identical, but more verbose.

In [ ]:
## GST Calculation using if/elif/else

def calculate_gst_if(item_category: str, base_price: float) -> dict:
    category = item_category.lower()

    if category == "essential":
        rate = 0.0
    elif category == "low":
        rate = 0.05
    elif category == "standard":
        rate = 0.12
    elif category == "higher":
        rate = 0.18
    elif category == "luxury":
        rate = 0.28
    else:
        print(f"Warning: Unknown category '{item_category}', defaulting to 18%")
        rate = 0.18

    gst_amount = base_price * rate
    total_price = base_price + gst_amount

    return {
        "category": item_category,
        "base_price": base_price,
        "gst_rate": f"{rate * 100:.0f}%",
        "gst_amount": round(gst_amount, 2),
        "total_price": round(total_price, 2)
    }


# Quick test
result = calculate_gst_if("Higher", 25000)
print(f"GST on ₹{result['base_price']:,.2f} ({result['category']}): "
      f"₹{result['gst_amount']:,.2f} → Total: ₹{result['total_price']:,.2f}")

# Learning Point: if/elif is evaluated top-to-bottom. Put the most common
# categories first if performance matters on millions of calls.

---
## Approach 3: Using a Dictionary Lookup (Most Elegant)

When mapping known keys to values, a **dictionary** is often cleaner than either `match` or `if/elif`.

**Time Complexity:** O(1) lookup — dictionaries use hash tables internally.

In [ ]:
## GST Calculation using dictionary lookup — cleanest approach

GST_RATES = {
    "essential": 0.00,
    "low":       0.05,
    "standard":  0.12,
    "higher":    0.18,
    "luxury":    0.28,
}

def calculate_gst_dict(item_category: str, base_price: float) -> dict:
    rate = GST_RATES.get(item_category.lower(), 0.18)  # Default 18% if unknown
    gst_amount = base_price * rate

    return {
        "category": item_category,
        "base_price": base_price,
        "gst_rate": f"{rate * 100:.0f}%",
        "gst_amount": round(gst_amount, 2),
        "total_price": round(base_price + gst_amount, 2)
    }


# Demonstrate with all categories
for cat, price in [("Essential", 200), ("Low", 500), ("Standard", 1200),
                    ("Higher", 25000), ("Luxury", 150000), ("Unknown", 1000)]:
    r = calculate_gst_dict(cat, price)
    print(f"{r['category']:>10} | ₹{r['base_price']:>10,.2f} + GST {r['gst_rate']:>3} "
          f"= ₹{r['total_price']:>10,.2f}")

# Best Practice: Dictionary lookup is O(1), self-documenting, and easy to maintain.
# If the GST rates come from a database or config file, this pattern scales naturally.

---
## Bonus: Price-Based Bracket Calculation

Sometimes the GST slab depends on the **price range** rather than a category label. This is where `match` with **guard clauses** shines.

In [ ]:
## Price-based GST using match with guards

def gst_by_price(price: float) -> float:
    """Determine GST rate based on price bracket."""
    match price:
        case p if p <= 0:
            raise ValueError("Price must be positive")
        case p if p <= 500:
            return 0.00       # Essential range
        case p if p <= 1000:
            return 0.05       # Low range
        case p if p <= 5000:
            return 0.12       # Standard range
        case p if p <= 50000:
            return 0.18       # Higher range
        case _:
            return 0.28       # Luxury range


# Test across price points
test_prices = [100, 750, 3000, 25000, 200000]

for price in test_prices:
    rate = gst_by_price(price)
    gst = price * rate
    print(f"₹{price:>8,.2f} → GST {rate*100:>2.0f}% = ₹{gst:>8,.2f} → Total: ₹{price + gst:>10,.2f}")

---
## Comparison: Which Approach When?

| Approach | Readability | Maintainability | Version Requirement |
|---|---|---|---|
| `match...case` | Clean for pattern matching | Good | Python 3.10+ |
| `if/elif/else` | Familiar to all developers | Verbose with many branches | Any Python |
| Dictionary lookup | Most concise | Easy to update rates | Any Python |
| `match` with guards | Best for range-based logic | Very readable | Python 3.10+ |

> **Perspective:** In production code, the GST rates would typically come from a database or configuration file — not hardcoded. The dictionary approach maps most naturally to that pattern. But for learning `match...case`, this is an ideal exercise.